# 30 — Complete Pipeline RAG

> **Tier 7 | Production | Capstone**

## What this notebook covers

This capstone notebook demonstrates a **production-grade multi-modal RAG pipeline** on
a rich PDF (`medicaid.pdf`) that contains:

| Content type | Count | Handling |
|---|---|---|
| Headings (L1–L4) | 21 ToC entries | `pymupdf get_toc()` + font-size fallback |
| Body text + checklist items | ~350 spans | `pymupdf get_text("dict")` with Wingdings mapping |
| Nested bullet lists | 3 indent levels | x-origin indent detection |
| Data tables | 5 real tables | `pdfplumber extract_table()` with ghost filter |
| Charts / figures | ~11 images | `pymupdf extract_image()` + Bedrock Claude vision |
| Dashboard screenshots | 2 full-page | Bedrock Claude vision |

### Pipeline stages
1. **Extract** — multi-library extraction, type-tagged chunks
2. **Index** — all chunk types into Qdrant with `chunk_type` payload
3. **Retrieve** — standard or `chunk_type`-filtered vector search
4. **Generate** — Claude Sonnet with context-aware system prompt
5. **Evaluate** — Hit@K, faithfulness, latency breakdown

## PDF Framework: multi-library capstone

| Library | Role |
|---------|------|
| `pymupdf get_toc()` | Heading hierarchy (21 ToC entries, L1–L4) |
| `pymupdf get_text("dict")` | Text / headings / bullets + Wingdings mapping |
| `pdfplumber extract_table()` | Structured data tables |
| `pymupdf extract_image()` + Bedrock vision | Chart and dashboard descriptions |


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph EXTRACT ["🔵  MULTI-MODAL EXTRACTION — medicaid.pdf"]
        PDF(["📄 medicaid.pdf"])
        PDF --> TOC["pymupdf get_toc()\nHeading hierarchy L1-L4"]
        PDF --> DICT["pymupdf get_text('dict')\nText + bullets + Wingdings"]
        PDF --> PL["pdfplumber extract_table()\nData tables (ghost filter)"]
        PDF --> IMG["pymupdf extract_image()\n+ Bedrock Claude vision"]
        TOC & DICT & PL & IMG --> TAG["Type-tagged chunks\ntext | heading | bullet | table | figure"]
    end
    subgraph INDEX ["🟡  INDEX"]
        TAG --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\ncomplete_pipeline_30")]
    end
    subgraph RETRIEVE ["🟢  RETRIEVE"]
        Q(["❓ Query"])
        Q --> VEC["Vector search"]
        VEC -->|optional| FILT["chunk_type filter\ntable / figure / text"]
        FILT --> QDRANT
        QDRANT --> CTX["Ranked context"]
    end
    subgraph GENERATE ["⚡  GENERATE + EVALUATE"]
        CTX --> LLM["Claude Sonnet"]
        LLM --> ANS(["✅ Answer"])
        ANS --> EVAL["Hit@K · Faithfulness · Latency"]
    end
    style EXTRACT fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style INDEX fill:#fef9c3,stroke:#ca8a04,color:#713f12
    style RETRIEVE fill:#dcfce7,stroke:#16a34a,color:#14532d
    style GENERATE fill:#fdf4ff,stroke:#9333ea,color:#3b0764
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "pymupdf", "pdfplumber", "python-dotenv", "Pillow"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, base64, re, math
from typing import List, Dict, Optional
from dotenv import load_dotenv

import boto3
import fitz          # pymupdf
import pdfplumber
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")
print(f"pymupdf  : {fitz.__version__}")
print(f"pdfplumber: {pdfplumber.__version__}")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

COLLECTION_NAME = "complete_pipeline_30"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
TOP_K           = 5

PDF_PATH = r"C:\Users\Administrator\RAG\data\medicaid.pdf"

# Image filtering
MIN_IMG_HEIGHT      = 50
DECORATIVE_WIDTHS   = {1030, 1170}

# Wingdings / Symbol glyph → readable text
WINGDING_MAP = {'\uf0fc': '[CHECK]', '\uf0fd': '[CROSS]', '\uf0fe': '[BOX]'}

print(f"Collection : {COLLECTION_NAME}")
print(f"PDF        : {PDF_PATH}  exists={os.path.exists(PDF_PATH)}")


## Step 3 — Qdrant Setup

In [ ]:
def make_qdrant(url='', api_key=''):
    if url:
        try:
            kw = {'url': url}
            if api_key: kw['api_key'] = api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {url}')
            return client
        except Exception as e:
            print(f'Cloud unavailable ({e}), falling back...')
    print('Using Qdrant in-memory')
    return QdrantClient(':memory:')


qdrant = make_qdrant(QDRANT_URL, QDRANT_API_KEY)

if COLLECTION_NAME in [c.name for c in qdrant.get_collections().collections]:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
print(f'Created "{COLLECTION_NAME}" (dim={EMBEDDING_DIM})')


## Step 4 — Bedrock Helpers (text + vision)

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)


def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]


def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out


def call_llm(system: str, user_content: str, max_tokens: int = 512) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user_content}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]


def call_llm_vision(img_bytes: bytes, media_type: str, prompt: str,
                    max_tokens: int = 250) -> str:
    b64 = base64.b64encode(img_bytes).decode()
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": media_type, "data": b64}},
                {"type": "text", "text": prompt}
            ]
        }]
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"].strip()


test_emb = embed_text("complete pipeline rag medicaid test")
print(f"Embed OK  — dim={len(test_emb)}")
print("Vision helper defined.")


## Step 5 — Multi-Modal PDF Extraction

Four extraction passes on the same PDF, each producing typed chunks.


In [ ]:
# ── Shared helpers ────────────────────────────────────────────────────────────

def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def map_wingdings(text: str) -> str:
    for glyph, repl in WINGDING_MAP.items():
        text = text.replace(glyph, repl)
    return text


def get_section(page_num: int, toc_sections: List[Dict]) -> str:
    """Return the ToC section title whose page <= page_num (last match)."""
    section = "Document"
    for s in toc_sections:
        if s['page'] <= page_num:
            section = s['title']
    return section


print("Shared helpers defined.")


### Pass A — ToC Hierarchy

In [ ]:
doc = fitz.open(PDF_PATH)
toc_raw = doc.get_toc()   # [(level, title, page), ...]
doc.close()

toc_sections = [{'level': lvl, 'title': title, 'page': pg}
                for lvl, title, pg in toc_raw]

print(f"ToC entries: {len(toc_sections)}")
for s in toc_sections:
    print(f"  {'  '*s['level']}[L{s['level']}] p{s['page']:>2}: {s['title']}")


### Pass B — Text / Headings / Bullets (`pymupdf get_text("dict")`)

Line classification rules:
- `size ≥ 13` → heading
- `size ≥ 11` + bold → heading  
- `x-origin ≥ 120` (indented bullet text) → bullet
- Wingdings `\uf0fc` → `[CHECK]` (checklist items)
- `size ≤ 9` or "Copyright" in text → footer, skip


In [ ]:
def extract_text_chunks(pdf_path: str) -> List[Dict]:
    doc = fitz.open(pdf_path)
    raw_lines = []

    for page_num in range(len(doc)):
        page    = doc[page_num]
        d       = page.get_text("dict")
        section = get_section(page_num + 1, toc_sections)

        for block in d.get("blocks", []):
            if block.get("type") != 0:
                continue
            for line in block.get("lines", []):
                x0       = line["bbox"][0]
                parts    = []
                max_size = 0
                is_bold  = False

                for span in line.get("spans", []):
                    t  = span["text"].strip()
                    if not t:
                        continue
                    s  = span["size"]
                    f  = span["font"]
                    fl = span["flags"]
                    bold = bool(fl & 16) or "Bold" in f

                    if "Wingding" in f or "Symbol" in f:
                        t = map_wingdings(t)

                    parts.append(t)
                    max_size = max(max_size, s)
                    is_bold  = is_bold or bold

                line_text = ' '.join(parts).strip()
                if not line_text:
                    continue
                # skip footers and page numbers
                if max_size <= 9.5 or "Copyright" in line_text:
                    continue
                if re.match(r'^\d{1,2}$', line_text):
                    continue

                # classify
                if max_size >= 13:
                    ctype = "heading"
                elif max_size >= 11 and is_bold:
                    ctype = "heading"
                elif x0 >= 118:
                    ctype = "bullet"
                else:
                    ctype = "text"

                raw_lines.append({
                    'text': line_text, 'type': ctype,
                    'page': page_num + 1, 'section': section, 'x0': x0,
                })
    doc.close()

    # Merge consecutive same-type lines on the same page
    merged  = []
    buf     = []
    buf_meta = {}

    def flush():
        if not buf:
            return
        text = ' '.join(buf).strip()
        for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
            merged.append({'text': sub, **buf_meta})
        buf.clear()

    for ln in raw_lines:
        same = (buf_meta.get('type') == ln['type'] and
                buf_meta.get('page') == ln['page'] and
                ln['type'] in ('text', 'bullet'))
        if same:
            buf.append(ln['text'])
        else:
            flush()
            if ln['type'] == 'heading':
                merged.append({'text': ln['text'], 'type': 'heading',
                               'page': ln['page'], 'section': ln['section']})
            else:
                buf.append(ln['text'])
                buf_meta = {'type': ln['type'], 'page': ln['page'],
                            'section': ln['section']}
    flush()
    return merged


t0 = time.time()
text_chunks = extract_text_chunks(PDF_PATH)
elapsed = (time.time() - t0) * 1000

type_counts = {}
for c in text_chunks:
    type_counts[c['type']] = type_counts.get(c['type'], 0) + 1

print(f"Text extraction : {elapsed:.0f}ms")
print(f"Total chunks    : {len(text_chunks)}")
for t, cnt in sorted(type_counts.items()):
    print(f"  {t:<10}: {cnt}")


### Pass C — Data Tables (`pdfplumber extract_table()` with ghost filter)

Ghost tables (layout artifacts) are detected by >50% None cells and skipped.
Real data table rows are converted to `"Header: value | Header: value"` strings.


In [ ]:
def is_ghost_table(table: List) -> bool:
    # Ghost if none of the data rows (row 1+) have any real content
    data_rows = table[1:]
    return not any(any(str(c or '').strip() for c in row) for row in data_rows)


def table_to_text(table: List) -> str:
    if not table or not table[0]:
        return ""
    header = [str(c or '').strip() for c in table[0]]
    lines  = []
    for row in table[1:]:
        cells = [str(c or '').replace('\n', ' ').strip() for c in row]
        if any(cells):
            pairs = ' | '.join(
                f"{h}: {v}" for h, v in zip(header, cells) if (h or v)
            )
            if pairs.strip():
                lines.append(pairs)
    return '\n'.join(lines) if lines else ' | '.join(header)


def extract_table_chunks(pdf_path: str) -> List[Dict]:
    chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            section = get_section(page_num + 1, toc_sections)
            for table in page.extract_tables():
                if is_ghost_table(table):
                    continue
                text = table_to_text(table)
                if text.strip():
                    chunks.append({
                        'text'   : f"[TABLE] {text}",
                        'type'   : 'table',
                        'page'   : page_num + 1,
                        'section': section,
                        'rows'   : len(table),
                        'cols'   : len(table[0]) if table else 0,
                    })
    return chunks


t0 = time.time()
table_chunks = extract_table_chunks(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"Table extraction : {elapsed:.0f}ms")
print(f"Real tables found: {len(table_chunks)}")
for c in table_chunks:
    print(f"  p{c['page']:>2} [{c['rows']}r x {c['cols']}c] {c['section'][:40]}: {c['text'][:80]}")


### Pass D — Figures & Charts (`pymupdf extract_image` + Bedrock Claude Vision)

Each non-decorative image is extracted as PNG bytes and sent to Claude Sonnet
for a 2–3 sentence description. That description becomes the figure's text chunk.


In [ ]:
VISION_PROMPT = (
    "Describe this chart, graph, or dashboard image in 2-3 sentences "
    "for a medical data visualization document. State the chart type, "
    "what data relationship it shows, and any key visual elements. "
    "If this is a logo, decorative line, or non-data element, reply with exactly: decorative"
)


def extract_image_png(doc, xref: int):
    """Extract image as PNG bytes regardless of original colorspace."""
    pix = fitz.Pixmap(doc, xref)
    if pix.n > 4:          # CMYK or special — convert to RGB
        pix = fitz.Pixmap(fitz.csRGB, pix)
    return pix.tobytes("png")


def extract_figure_chunks(pdf_path: str) -> List[Dict]:
    doc    = fitz.open(pdf_path)
    chunks = []

    for page_num in range(len(doc)):
        page    = doc[page_num]
        section = get_section(page_num + 1, toc_sections)
        imgs    = page.get_images(full=True)
        fig_idx = 0

        for img_info in imgs:
            xref = img_info[0]
            w, h = img_info[2], img_info[3]

            # skip decorative elements
            if h < MIN_IMG_HEIGHT or w in DECORATIVE_WIDTHS:
                continue

            try:
                img_bytes   = extract_image_png(doc, xref)
                description = call_llm_vision(img_bytes, "image/png", VISION_PROMPT)

                if description.strip().lower() == "decorative":
                    print(f"  p{page_num+1} xref={xref} ({w}×{h}): decorative — skipped")
                    continue

                fig_idx += 1
                label = f"[FIGURE] Page {page_num+1}, Figure {fig_idx} ({w}x{h}px): {description}"
                chunks.append({
                    'text'    : label,
                    'type'    : 'figure',
                    'page'    : page_num + 1,
                    'section' : section,
                    'img_size': f"{w}x{h}",
                })
                print(f"  p{page_num+1} fig{fig_idx} ({w}x{h}): {description[:90]}")
                time.sleep(0.2)

            except Exception as e:
                print(f"  p{page_num+1} xref={xref}: error — {e}")

    doc.close()
    return chunks


print("Extracting figures (Claude vision)...")
t0 = time.time()
figure_chunks = extract_figure_chunks(PDF_PATH)
elapsed = (time.time() - t0) * 1000
print(f"\nFigure extraction : {elapsed:.0f}ms")
print(f"Figure chunks     : {len(figure_chunks)}")


### Extraction Summary

In [ ]:
all_chunks = text_chunks + table_chunks + figure_chunks

print("Extraction summary")
print("-" * 40)
print(f"Text/heading/bullet : {len(text_chunks)}")
print(f"Table chunks        : {len(table_chunks)}")
print(f"Figure chunks       : {len(figure_chunks)}")
print(f"Total chunks        : {len(all_chunks)}")
print()
print("By type:")
type_summary = {}
for c in all_chunks:
    type_summary[c['type']] = type_summary.get(c['type'], 0) + 1
for t, cnt in sorted(type_summary.items()):
    print(f"  {t:<10}: {cnt:>3} chunks")


## Step 6 — Index All Chunk Types

In [ ]:
from qdrant_client.models import PayloadSchemaType

print(f"Embedding {len(all_chunks)} chunks...")
t0   = time.time()
texts = [c['text'] for c in all_chunks]
embs  = embed_batch(texts, label='[index]')

pts = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=embs[i],
        payload={
            'text'      : all_chunks[i]['text'],
            'chunk_type': all_chunks[i]['type'],
            'page'      : all_chunks[i]['page'],
            'section'   : all_chunks[i]['section'],
            'source'    : 'medicaid.pdf',
        }
    )
    for i in range(len(all_chunks))
]
qdrant.upsert(collection_name=COLLECTION_NAME, points=pts)

# Create keyword index on chunk_type to enable filtered queries
qdrant.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="chunk_type",
    field_schema=PayloadSchemaType.KEYWORD,
)

n_indexed = qdrant.get_collection(COLLECTION_NAME).points_count
print(f"Indexed {n_indexed} vectors in {time.time()-t0:.1f}s")
print(f"Payload index created on 'chunk_type' (keyword)")


## Step 7 — Complete RAG Pipeline

Supports optional `chunk_type` filtering — e.g. search only among tables or figures.


In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant for a medical data visualization document.
Answer ONLY from the provided context. Be concise and precise.
If the answer is not in the context, say so explicitly.
"""


def complete_rag(question: str,
                chunk_type_filter: Optional[str] = None,
                verbose: bool = True) -> Dict:
    t0 = time.time()

    # 1. Embed
    qvec    = embed_text(question)
    t_embed = (time.time() - t0) * 1000

    # 2. Retrieve with optional type filter
    qfilter = None
    if chunk_type_filter:
        qfilter = Filter(
            must=[FieldCondition(
                key="chunk_type",
                match=MatchValue(value=chunk_type_filter)
            )]
        )

    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=qvec,
        limit=TOP_K,
        with_payload=True,
        query_filter=qfilter,
    )
    hits     = [{
        'text'   : p.payload['text'],
        'type'   : p.payload.get('chunk_type', '?'),
        'page'   : p.payload.get('page', '?'),
        'section': p.payload.get('section', '?'),
        'score'  : p.score,
    } for p in resp.points]
    t_ret = (time.time() - t0) * 1000 - t_embed

    # 3. Build context
    context = '\n\n'.join(
        f"[{i+1}] type={h['type']} section={h['section']} page={h['page']}\n{h['text']}"
        for i, h in enumerate(hits)
    )

    # 4. Generate
    answer  = call_llm(SYSTEM_PROMPT,
                       f"Context:\n{context}\n\nQuestion: {question}")
    t_total = (time.time() - t0) * 1000

    if verbose:
        fstr = f" [filter='{chunk_type_filter}']" if chunk_type_filter else ""
        print(f"Q: {question}")
        print(f"   chunks={len(hits)}{fstr}  embed={t_embed:.0f}ms  retrieve={t_ret:.0f}ms  total={t_total:.0f}ms")
        for h in hits[:3]:
            print(f"   [{h['type']:>7}] p{h['page']} score={h['score']:.3f}: {h['text'][:80]}")
        print(f"A: {answer[:350]}")
        print()

    return {
        'question': question, 'answer': answer,
        'hits': hits, 'filter': chunk_type_filter,
        'embed_ms': t_embed, 'retrieve_ms': t_ret, 'total_ms': t_total,
    }


print("complete_rag() defined.")


## Step 8 — Demo: Multi-Modal Query Results

Four queries demonstrating different retrieval paths:
1. General text query (no filter)
2. Table-only query (filter=table)
3. Figure/chart query (filter=figure)
4. Dashboard query (free retrieval from figure chunks)


In [ ]:
print("=" * 70)
print("DEMO 1 — General text retrieval (no filter)")
print("=" * 70)
r1 = complete_rag(
    "What are the best practices for table organization and categorization?",
    verbose=True
)


In [ ]:
print("=" * 70)
print("DEMO 2 — Table-filtered retrieval")
print("=" * 70)
r2 = complete_rag(
    "What chart types should be used for nominal comparison and ranking relationships?",
    chunk_type_filter="table",
    verbose=True
)


In [ ]:
print("=" * 70)
print("DEMO 3 — Figure-filtered retrieval")
print("=" * 70)
r3 = complete_rag(
    "What types of charts and graphs are shown in the document examples?",
    chunk_type_filter="figure",
    verbose=True
)


In [ ]:
print("=" * 70)
print("DEMO 4 — Mixed retrieval (heading + text + table)")
print("=" * 70)
r4 = complete_rag(
    "What is a data dashboard and what are the key principles for creating one?",
    verbose=True
)


## Step 9 — Evaluation

In [ ]:
JUDGE_SYSTEM  = "You are a strict evaluator of RAG system quality. Return valid JSON only."
FAITH_TMPL = """Rate faithfulness 1-5 (5=all claims supported by context).
Context: {context}\nAnswer: {answer}\nReturn ONLY JSON: {{"score":<1-5>,"reason":"<one sentence>"}}"""
RELEV_TMPL = """Rate answer relevance 1-5 (5=fully addresses question).
Question: {question}\nAnswer: {answer}\nReturn ONLY JSON: {{"score":<1-5>,"reason":"<one sentence>"}}"""

def judge(raw):
    try:
        s, e = raw.find('{'), raw.rfind('}') + 1
        return json.loads(raw[s:e])
    except Exception:
        return {'score': 0, 'reason': 'parse error'}

all_results = [r1, r2, r3, r4]
eval_results = []

print("Running LLM evaluation on 4 demo queries...")
for r in all_results:
    context = '\n'.join(h['text'][:300] for h in r['hits'])
    faith = judge(call_llm(JUDGE_SYSTEM, FAITH_TMPL.format(
        context=context[:2000], answer=r['answer']), max_tokens=100))
    relev = judge(call_llm(JUDGE_SYSTEM, RELEV_TMPL.format(
        question=r['question'], answer=r['answer']), max_tokens=100))
    eval_results.append({'faithfulness': faith['score'], 'relevance': relev['score']})
    flt = f" [{r['filter']}]" if r['filter'] else ""
    print(f"  {r['question'][:55]}{flt}")
    print(f"     faith={faith['score']}/5  relevance={relev['score']}/5")

avg_faith = sum(e['faithfulness'] for e in eval_results) / len(eval_results)
avg_relev = sum(e['relevance']    for e in eval_results) / len(eval_results)
print(f"\nAvg Faithfulness : {avg_faith:.2f}/5")
print(f"Avg Relevance    : {avg_relev:.2f}/5")


## Step 10 — Summary Report

In [ ]:
def avg(lst): return sum(lst)/len(lst) if lst else 0

print("=" * 62)
print("COMPLETE PIPELINE RAG — Notebook 30 Summary")
print("=" * 62)
print(f"PDF              : medicaid.pdf (10 pages, rich multi-modal)")
print(f"Collection       : {COLLECTION_NAME}")
print(f"Vectors indexed  : {qdrant.get_collection(COLLECTION_NAME).points_count}")
print()
print("Extraction breakdown")
print("-" * 35)
for t, cnt in sorted(type_summary.items()):
    print(f"  {t:<10}: {cnt:>3} chunks")
print(f"  {'TOTAL':<10}: {len(all_chunks):>3} chunks")
print()
print("Extraction libraries")
print("-" * 35)
print("  pymupdf get_toc()         — heading hierarchy (21 ToC entries)")
print("  pymupdf get_text('dict')  — text / headings / bullets + Wingdings")
print("  pdfplumber extract_table()— data tables (ghost-filtered)")
print("  pymupdf + Bedrock vision  — chart & dashboard descriptions")
print()
print("Generation quality (LLM judge)")
print("-" * 35)
print(f"  Avg Faithfulness : {avg_faith:.2f}/5")
print(f"  Avg Relevance    : {avg_relev:.2f}/5")
print()
print("Latency (avg over 4 queries)")
print("-" * 35)
print(f"  Embed    : {avg([r['embed_ms']    for r in all_results]):.0f}ms")
print(f"  Retrieve : {avg([r['retrieve_ms'] for r in all_results]):.0f}ms")
print(f"  Total    : {avg([r['total_ms']    for r in all_results]):.0f}ms")
print()
print("Notebook 30 — Complete Pipeline RAG complete.")


## Summary

### Multi-modal extraction pipeline

| Pass | Library | Content type | Key technique |
|------|---------|--------------|---------------|
| A | `pymupdf get_toc()` | Heading hierarchy | 21 embedded bookmarks, L1–L4 |
| B | `pymupdf get_text("dict")` | Text / headings / bullets | Font-size + x-origin indent + Wingdings map |
| C | `pdfplumber extract_table()` | Structured tables | Ghost-filter (>50% None), row→text conversion |
| D | `pymupdf` + Bedrock Claude vision | Charts / dashboards | `extract_image()` → base64 PNG → LLM description |

### chunk_type-filtered retrieval

```python
# Retrieve only table chunks
qdrant.query_points(
    collection_name=COLLECTION_NAME, query=qvec, limit=5,
    query_filter=Filter(must=[FieldCondition(
        key="chunk_type", match=MatchValue(value="table")
    )])
)
```

### Wingdings checkbox handling
```python
WINGDING_MAP = {'\uf0fc': '[CHECK]', '\uf0fd': '[CROSS]'}
# Detect: span['font'] contains 'Wingding' or 'Symbol'
# Map before embedding to avoid garbage tokens
```

### When to use each library

| Requirement | Best choice |
|---|---|
| Clean body text | `pymupdf get_text("text")` |
| Headings + font metadata | `pymupdf get_text("dict")` |
| Bordered/data tables | `pdfplumber extract_table()` |
| Charts / figures | `pymupdf extract_image()` + Bedrock vision |
| Scanned PDFs | AWS Textract |
